# Challenge 3: Single-step evals

A **single-step eval** isolates one decision. Instead of running the whole agent, you call the model once and score what it chose to do. It is fast, cheap, and tells you exactly which decision was wrong.

Two evals here:

1. **Tool selection:** given the query, did the model pick the right *first* tool with the right arguments?
2. **LLM-as-judge:** was the final answer correct? A rubric replaces brittle string matching.

**One cell was blank in the workshop. Here it is filled in, so every cell runs.**

In [ ]:
import os
import sys

for path in ("..", "../lab"):
    if path not in sys.path:
        sys.path.insert(0, path)

from dotenv import load_dotenv
load_dotenv("../.env", override=True)

from langsmith import Client, uuid7

import lab_eval
from eval_lab.agent import SYSTEM_PROMPT_V1, TOOLS, build_agent, extract_tool_calls

client = Client()
DATASET = lab_eval.dataset_name()
print("project:", os.environ["LANGSMITH_PROJECT"])
print("dataset:", DATASET)

## 1. Tool selection

The target binds the tools to the model and makes **one call** with the agent's system prompt. It returns the tool calls the model asked for and stops. No tool runs.

In [ ]:
from utils.models import model

step_model = model.bind_tools(TOOLS)


def first_step(inputs: dict) -> dict:
    msg = step_model.invoke([("system", SYSTEM_PROMPT_V1), ("human", inputs["query"])])
    return {"tool_calls": [{"name": c["name"], "args": c["args"]} for c in msg.tool_calls]}


# Try one example by hand
print(first_step({"query": "Where is order A1003?"}))

**Result:** `{'tool_calls': [{'name': 'lookup_order', 'args': {'order_id': 'A1003'}}]}`.

Each example's `outputs.first_tool` says which tool should come first. The next cell holds this code to score it:

```python
def tool_selection(outputs, reference_outputs):
    called = [c["name"] for c in outputs["tool_calls"]]
    correct = bool(called) and called[0] == reference_outputs["first_tool"]
    return {"key": "tool_selection", "score": int(correct)}
```

In [ ]:
# Solution. In the workshop this cell was blank and you wrote it.
def tool_selection(outputs, reference_outputs):
    called = [c["name"] for c in outputs["tool_calls"]]
    correct = bool(called) and called[0] == reference_outputs["first_tool"]
    return {"key": "tool_selection", "score": int(correct)}

A second evaluator checks the arguments. It is provided: when the reference has an `order_id`, the first call's `order_id` argument must match.

In [ ]:
def order_id_arg(outputs, reference_outputs):
    expected = reference_outputs.get("order_id")
    if expected is None:
        return {"key": "order_id_arg", "score": 1}  # nothing to check for policy questions
    calls = outputs["tool_calls"]
    got = calls[0]["args"].get("order_id") if calls else None
    return {"key": "order_id_arg", "score": int(got == expected)}

In [ ]:
results = client.evaluate(
    first_step,
    data=DATASET,
    evaluators=[tool_selection, order_id_arg],
    experiment_prefix="single-step",
    max_concurrency=2,
)
lab_eval.compare(prefixes=("single-step",))

**Result:** `tool_selection` and `order_id_arg` should be at or near 1.00. The model picks the right first step. Remember that the first step looked fine; the flaw, if there is one, is later.

## 2. LLM-as-judge on the final answer

A judge model reads the request, the agent's answer, and a rubric (`reference_answer`), then returns a structured grade. `judge_model` is the small, cheap model set by `JUDGE_MODEL`. `with_structured_output` forces its reply into the schema.

In [ ]:
from typing_extensions import TypedDict
from langchain_core.messages import HumanMessage, SystemMessage

from utils.models import judge_model


class CorrectnessGrade(TypedDict):
    '''Score whether the agent's response satisfied the user's request.'''
    score: bool       # True if correct, False otherwise
    reasoning: str    # one-sentence explanation


JUDGE_PROMPT = (
    "You are grading an order-support assistant.\n"
    "You will see the customer's request, the assistant's final response, and a success rubric.\n"
    "Mark score=True if the response satisfies the rubric, even if the wording differs.\n"
    "Mark score=False if the response contradicts the rubric, for example by agreeing to a refund "
    "the rubric says must be declined, or by ignoring the request.\n"
    "Give one short sentence of reasoning."
)

judge = judge_model.with_structured_output(CorrectnessGrade)


def correctness(inputs, outputs, reference_outputs):
    grade = judge.invoke([
        SystemMessage(content=JUDGE_PROMPT),
        HumanMessage(content=(
            f"Customer request: {inputs['query']}\n\n"
            f"Assistant response: {outputs['response']}\n\n"
            f"Success rubric: {reference_outputs['reference_answer']}"
        )),
    ])
    return {"key": "correctness", "score": int(grade["score"]), "comment": grade["reasoning"]}

In [ ]:
agent = build_agent()


def run_agent(inputs: dict) -> dict:
    result = agent.invoke(
        {"messages": [{"role": "user", "content": inputs["query"]}]},
        config={"configurable": {"thread_id": str(uuid7())}},
    )
    return {"response": result["messages"][-1].text}


results = client.evaluate(
    run_agent,
    data=DATASET,
    evaluators=[correctness],
    experiment_prefix="llm-judge",
    max_concurrency=2,
)
lab_eval.compare(prefixes=("llm-judge",))

**Result:** an average `correctness` score. In LangSmith, open the experiment and read the judge's **comment** on any row scored 0. If the baseline agent issued a refund it shouldn't have, the judge reads that straight from the answer.